# Repository code quality

Run all cells with a Windows Python kernel containing `radon`. This notebook calls `scripts/run_quality_checks.bat` using that kernel's Python. Text reports and separate stderr logs are written to `outputs/quality/`.

For lint, docstrings, security, tests, and coverage, run `scripts/run_code_checks.bat` separately. Neither script automatically edits source files or installs dependencies.

In [5]:
from pathlib import Path
import os
import re
from collections import Counter
import subprocess  # nosec B404

## Report parsing and scoring

Missing or malformed reports raise an error. Maintainability uses the mean of the numeric Radon indices, not an invented score from letter grades. The combined score is a heuristic with 45% complexity, 45% maintainability, and 10% comment-density weights; it is not comparable to the earlier grade-weighted score.

In [6]:
MAX_COMPLEXITY = 15
COMPLEXITY_FACTOR = 0.45
MAINTAINABILITY_FACTOR = 0.45
COMMENT_FACTOR = 0.1

def parse_cyclomatic_complexity(file_path: str) -> tuple[int, float, list[str]]:
    with open(file_path, "r", encoding="utf-8") as f:
        content = f.read()

    # Extract average complexity
    avg_match = re.search(r"Average complexity: [A-Z] \(([\d.]+)\)", content)
    avg_score = float(avg_match.group(1)) if avg_match else 0.0

    # Extract number of analyzed blocks
    total_match = re.search(r"(\d+)\s+blocks \(classes, functions, methods\) analyzed\.", content)
    total_blocks = int(total_match.group(1)) if total_match else None

    flagged_with_scores = []
    for line in content.splitlines():
        match = re.search(r"- [A-Z] \(([\d.]+)\)", line)
        if match:
            score = float(match.group(1))
            if score > MAX_COMPLEXITY:
                flagged_with_scores.append((score, line.strip()))

    flagged = [line for _, line in sorted(flagged_with_scores, key=lambda item: item[0], reverse=True)]

    return total_blocks, avg_score, flagged


def parse_maintainability(file_path: str) -> tuple[list[str], float, dict[str, float]]:
    grade_weights = {"A": 3, "B": 2, "C": 1}
    grade_counts = Counter()
    total_files = 0
    flagged = []


    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            match = re.search(r"([^\s]+\.py) - ([A-Z]) \(([\d.]+)\)", line)
            if match:
                file_name = match.group(1)
                grade = match.group(2)
                score = float(match.group(3))
                grade_counts[grade] += 1
                total_files += 1
                if grade != "A":
                    flagged.append((score, line.strip()))

    if total_files == 0:
        return [], 0.0, {}

    # Calculate weighted index (normalized to 0–100)
    weighted_index = (
        sum(grade_weights.get(grade, 0) * count for grade, count in grade_counts.items())
        / (3 * total_files)
    ) * 100

    grade_distribution = {
        grade: round((count / total_files) * 100, 2) for grade, count in grade_counts.items()
    }

    sorted_flagged = [entry for _, entry in sorted(flagged, key=lambda item: item[0])]
    return sorted_flagged, round(weighted_index, 2), grade_distribution


def classify_comment_density(density: float) -> str:
    """Classify comment density into qualitative labels.

    Parameters
    ----------
    density : float
        Comment density as a percentage of LOC.

    Returns
    -------
    str
        Qualitative label.
    """
    if density >= 50:
        return "🏆 Excellent"
    elif 35 <= density < 50:
        return "✅ Very Good"
    elif 25 <= density < 35:
        return "👍 Good"
    elif 15 <= density < 25:
        return "⚠️ Fair"
    else:
        return "🚨 Poor"


def normalized_comment_score(comment_avg: float) -> float:
    """Convert comment density to a normalized score based on tiers."""
    if comment_avg >= 50:
        # Map 50–100 → 90–100
        return 90 + (comment_avg - 50) * (10 / 50)
    elif 35 <= comment_avg < 50:
        return 75 + (comment_avg - 35) * (15 / 15)
    elif 25 <= comment_avg < 35:
        return 60 + (comment_avg - 25) * (15 / 10)
    elif 15 <= comment_avg < 25:
        return 40 + (comment_avg - 15) * (20 / 10)
    else:
        return comment_avg * (39 / 15)  # Map 0–15 → 0–39


def parse_comment_density(file_path: str) -> tuple[list[str], float, str]:
    densities = []
    flagged = []
    current_file = None

    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip().endswith(".py"):
                current_file = line.strip()
            match = re.search(r"\(C \+ M % L\): (\d+)%", line)
            if match:
                pct = int(match.group(1))
                densities.append(pct)
                if pct < 25 and current_file:
                    flagged.append((pct, f"{current_file}: {pct}%"))
    avg_density = sum(densities) / len(densities) if densities else 0.0
    density_label = classify_comment_density(avg_density)
    sorted_flagged = [entry for _, entry in sorted(flagged, key=lambda item: item[0])]
    return sorted_flagged, avg_density, density_label


def calculate_code_quality_score(complexity: float, mi_avg: float, comment_avg: float) -> float:
    complexity_score = max(0.0, 100 - complexity * 10)
    comment_score = normalized_comment_score(comment_avg)
    return round((mi_avg * MAINTAINABILITY_FACTOR + comment_score * COMMENT_FACTOR + complexity_score * COMPLEXITY_FACTOR) / (MAINTAINABILITY_FACTOR + COMMENT_FACTOR + COMPLEXITY_FACTOR), 2)


def generate_code_quality_report(
    path_complexity: str,
    path_maintainability: str,
    path_raw_metrics: str
) -> None:
    print("📊 Generating Code Quality Report...\n")

    # Parse all files
    total_blocks, avg_complexity, complex_funcs = parse_cyclomatic_complexity(path_complexity)
    mi_flags, mi_avg, mi_distribution = parse_maintainability(path_maintainability)
    low_doc_files, comment_avg, comment_label = parse_comment_density(path_raw_metrics)

    # Calculate score
    cqs = calculate_code_quality_score(avg_complexity, mi_avg, comment_avg)
    if not total_blocks:
        raise RuntimeError("Error: there aren't any functions checked.")

    # Print section-wise summary
    print("✅ 1. Cyclomatic Complexity (from cyclomatic_complexity.txt)")
    print(f"Total functions/methods analyzed: {total_blocks}")
    print(f"Average complexity score: A ({avg_complexity:.2f}) → {'Excellent' if avg_complexity <= MAX_COMPLEXITY else 'Needs Refactor'}\n")
    if complex_funcs:
        print(f"Functions with complexity score > {MAX_COMPLEXITY} (highest first):")
        for line in complex_funcs[:20]:
            print(f"  - {line}")
        if len(complex_funcs) > 20:
            print("  ...")

    print("\n✅ 2. Maintainability Index (from maintainability_index.txt)")
    print(f"Average maintainability index: {mi_avg:.2f}")
    print(f"Maintainability distribution: {mi_distribution}")

    if mi_flags:
        print("Modules with maintainability issues:")
        for line in mi_flags:
            print(f"  - {line}")
    else:
        print("All files are well-maintained (A ratings).")

    print("\n✅ 3. Raw Metrics and Documentation (from raw_metrics.txt)")
    print(f"Average comment density: {comment_avg:.2f}% → {comment_label}")
    if low_doc_files:
        print("Low-commented files (<25%):")
        for line in low_doc_files:
            print(f"  - {line}")
    else:
        print("All files have acceptable documentation levels.")

    print("\n🎯 Final Code Quality Score: ", f"{cqs} / 100")


In [7]:
def run_quality_checks(batch_path: str) -> None:
    """Run the Radon batch script and ensure it completes successfully."""
    batch_file = Path(batch_path)
    if not batch_file.is_file():
        raise FileNotFoundError(f"Batch file not found: {batch_file}")

    print(f"🔧 Running: {batch_file} ...")
    result = subprocess.run(
        [str(batch_file)],
        check=True, shell=False,
        cwd=batch_file.parent,  # Ensures paths in BAT are resolved correctly
        capture_output=True,
        text=True,
        timeout=3600,
    )

    if result.returncode != 0:
        print("❌ Error running batch file:\n", result.stderr)
        raise RuntimeError("Batch execution failed.")

    print("✅ Batch file executed successfully.\n")
    print(result.stdout)

In [8]:
from pathlib import Path

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
bat_path = str(REPO / "scripts")

print(bat_path)
txt_path = str(REPO / "outputs" / "quality")
file_cyclomatic_complexity = os.path.join(txt_path, "cyclomatic_complexity.txt")
file_maintainability_index = os.path.join(txt_path, "maintainability_index.txt")
file_raw_metrics = os.path.join(txt_path, "raw_metrics.txt")

run_quality_checks(os.path.join(bat_path, "run_quality_checks.bat"))

generate_code_quality_report(
    file_cyclomatic_complexity,
    file_maintainability_index,
    file_raw_metrics
)

c:\work_dir\satellites\scripts
🔧 Running: c:\work_dir\satellites\scripts\run_quality_checks.bat ...


✅ Batch file executed successfully.

Python: c:\Users\e_louvari\.conda\envs\eo_spatial\python.exe
Measuring cyclomatic complexity...
Measuring maintainability...
Measuring raw metrics...
Reports: c:\work_dir\satellites\outputs\quality
Quality reports generated.

📊 Generating Code Quality Report...

✅ 1. Cyclomatic Complexity (from cyclomatic_complexity.txt)
Total functions/methods analyzed: 1314
Average complexity score: A (3.94) → Excellent


✅ 2. Maintainability Index (from maintainability_index.txt)
Average maintainability index: 98.02
Maintainability distribution: {'A': 94.07, 'B': 5.93}
Modules with maintainability issues:
  - src\ml_classification\shared\reports\confidence_diagnostics.py - B (14.24)
  - src\shared\raster.py - B (14.62)
  - src\ml_classification\shared\reports\final_dashboard_artifact.py - B (14.85)
  - src\eda\core\profiling_statistics.py - B (14.96)
  - src\data_preparation\parcel_stats\core\raster_cleaning.py - B (15.63)
  - src\data_preparation\sources\hub\cat